[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S07/2_Practica/para_casa/S07_01_regresion_lineal_polinomica.ipynb)

# S07 · 01 — Regresión lineal y polinómica: predecir el precio por noche

Sprint 7 · M3 (UD 3.3)

**Contexto TurisData.** El cliente quiere una tarifa de referencia: *¿cuánto debería costar la noche de una reserva según el tipo de habitación y la época?* Es un problema de **regresión** (predecir un número). Empezamos por el modelo más sencillo y más explicable: la regresión lineal.

**Este cuaderno tiene 4 ejercicios y un mini-reto.** Los ejercicios 1, 2 y 3 son los mismos que los del cuaderno «en clase» (allí, del 1 al 3).

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEMILLA = 42
sns.set_theme(style="whitegrid")

df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
df["mes_llegada"] = df["fecha_llegada"].dt.month
df["temporada_alta"] = df["mes_llegada"].isin([1, 2, 3, 7, 8, 12]).astype(int)
print(df.shape)
print("Listo ✔")

### 👀 A2 · EJEMPLO — Preparar X e y con variables dummy

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Cuántas columnas tiene `X` y cuáles son?
2. ¿Qué habitación es la «referencia» y cómo se reconoce en la tabla?
3. ¿Qué reserva es la fila 3?

In [ ]:
X = pd.get_dummies(df[["tipo_habitacion"]], drop_first=True).astype(int)
X["temporada_alta"] = df["temporada_alta"]
y = df["precio_noche"]
X.head()

### ✏️ A3 · EJERCICIO 1 de 4 — Ajustar la regresión lineal

**Tu turno:** Separa en `X_train, X_test, y_train, y_test` (20 % de test, `random_state=SEMILLA`), ajusta una `LinearRegression` llamada `modelo_lin` y guarda en `r2_test` y `mae_test` sus métricas en test.

**Debe salir:** `R² = 0.915 · MAE = 7.24 € · RMSE = 9.70 €` y ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A3).

1. ¿Qué valores salen de R², MAE y RMSE?
2. ¿Qué significa un R² de 0,915?
3. ¿Por qué el RMSE (9,70) es mayor que el MAE (7,24)?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A3, **después de intentarlo**.

In [ ]:
# pista: train_test_split(X, y, test_size=0.2, random_state=SEMILLA); LinearRegression().fit(...); .predict(X_test)
X_train, X_test, y_train, y_test = ...
modelo_lin = ...
pred = ...
r2_test = ...
mae_test = ...

assert len(X_test) == 1200, "Con 6000 filas y test_size=0.2 el test debe tener 1200 filas"
assert r2_test > 0.85, "R² debería superar 0,85: revisa que uses tipo de habitación y temporada"
assert 5 < mae_test < 15, "El MAE debería estar entre 5 y 15 euros"
print(f"R² = {r2_test:.3f} · MAE = {mae_test:.2f} € · RMSE = {mean_squared_error(y_test, pred) ** 0.5:.2f} €")
print("✔ Ejercicio 1 correcto")

### 👀 A4 · EJEMPLO — El baseline: predecir siempre la media

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A4).

1. ¿Qué MAE y qué R² tiene el baseline?
2. ¿Por qué su R² es ≈ 0?
3. ¿Cuánto mejora el modelo lineal frente al baseline?

In [ ]:
base = DummyRegressor(strategy="mean").fit(X_train, y_train)
print(f"Baseline: MAE = {mean_absolute_error(y_test, base.predict(X_test)):.2f} € · R² = {r2_score(y_test, base.predict(X_test)):.3f}")

### ✏️ A5 · EJERCICIO 2 de 4 — Leer los coeficientes en euros

**Tu turno:** Consulta `modelo_lin.coef_` (van en el mismo orden que las columnas de `X`). Guarda en `coefs` un `pd.Series` con los coeficientes y el nombre de columna como índice, y responde con números: `extra_suite` (euros extra de una suite frente a la estándar) y `efecto_temporada_alta` (euros extra por reservar en temporada alta).

**Debe salir:** la tabla de coeficientes (`familiar 45.0`, `suite 101.2`, `superior 28.4`, `temporada_alta 22.4`), `Ordenada en el origen (lo que predice el modelo para una estándar en temporada baja): 67.3 €` y ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A5).

1. ¿Cuánto valen `extra_suite` y `efecto_temporada_alta`?
2. ¿Qué significa la ordenada en el origen, 67,3?
3. Según el modelo, ¿cuánto costaría una familiar en temporada alta?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A5, **después de intentarlo**.

In [ ]:
# pista: pd.Series(modelo_lin.coef_, index=X.columns); luego coefs["nombre_de_columna"]
coefs = ...
extra_suite = ...
efecto_temporada_alta = ...

assert 90 < extra_suite < 115, "Una suite cuesta unos 100 € más que una estándar (coeficiente de tipo_habitacion_suite)"
assert 15 < efecto_temporada_alta < 30, "La temporada alta añade unos 20-25 €"
print(coefs.round(1))
print("Ordenada en el origen (lo que predice el modelo para una estándar en temporada baja):", round(modelo_lin.intercept_, 1), "€")
print("✔ Ejercicio 2 correcto")

### 👀 A6 · EJEMPLO — Residuos del modelo lineal

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A6).

1. ¿Qué forma tiene la nube de la izquierda?
2. ¿Dónde se equivoca más el modelo, en euros: en las estándar o en las suites?
3. ¿Cómo es la distribución de la derecha?

In [ ]:
residuos = y_test - pred
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].scatter(pred, residuos, s=10, alpha=0.4, color="#2a6f97")
ejes[0].axhline(0, color="black", lw=1)
ejes[0].set(title="Residuos frente a predicción", xlabel="Precio predicho (€)", ylabel="Residuo (€)")
sns.histplot(residuos, bins=30, ax=ejes[1], color="#2a6f97")
ejes[1].set(title="Distribución de los residuos", xlabel="Residuo (€)", ylabel="Reservas")
plt.tight_layout()
plt.show()

### 👀 A7 · EJEMPLO — Interacciones con validación cruzada

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A7).

1. ¿Qué R² medio sale con cada modelo?
2. ¿Cuánto mejora el R² y qué te parece?
3. ¿Qué permite una interacción como `suite × temporada_alta`?

In [ ]:
lineal = LinearRegression()
con_interacciones = make_pipeline(PolynomialFeatures(degree=2, interaction_only=True, include_bias=False),
                                  LinearRegression())
r2_lineal_cv = cross_val_score(lineal, X, y, cv=5, scoring="r2").mean()
r2_inter_cv = cross_val_score(con_interacciones, X, y, cv=5, scoring="r2").mean()
print(f"R² CV lineal: {r2_lineal_cv:.3f} · con interacciones: {r2_inter_cv:.3f}")

### ✏️ A8 · EJERCICIO 3 de 4 — Error de las interacciones en test

**Tu turno:** Calcula `mae_inter_test`: el MAE en test del modelo `con_interacciones` ajustado con `X_train, y_train`. Guarda en `mejora_mae` cuántos euros mejora respecto a `mae_test` (positivo = mejor).

**Debe salir:** `MAE sin interacciones: 7.24 € · con interacciones: 6.76 € · mejora: 0.48 €` y ✔ Ejercicio 3 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A8).

1. ¿Cuánto valen `mae_inter_test` y `mejora_mae`?
2. ¿Es mucha mejora? Compárala con el MAE del modelo lineal (7,24 €).
3. ¿Por qué se ajusta con `X_train` y se mide el error en `X_test`?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A8, **después de intentarlo**.

In [ ]:
# pista: ajusta con_interacciones con train y predice sobre test
mae_inter_test = ...
mejora_mae = ...

assert mae_inter_test < mae_test, "Las interacciones deberían reducir el error"
assert mejora_mae > 0.3, "La mejora debería ser apreciable (más de 0,3 €)"
print(f"MAE sin interacciones: {mae_test:.2f} € · con interacciones: {mae_inter_test:.2f} € · mejora: {mejora_mae:.2f} €")
print("✔ Ejercicio 3 correcto")

### 👀 A9 · EJEMPLO — Polinomios de grado 1 a 12

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A9).

1. ¿Qué R² tienen el polinomio de grado 2 y el de grado 8?
2. ¿Por qué un polinomio de grado 2 no capta la relación entre el mes y el precio?
3. ¿Qué pasa con el R² al subir el grado de 7 a 12?

In [ ]:
est = df[df["tipo_habitacion"] == "estandar"]
X_mes, y_est = est[["mes_llegada"]], est["precio_noche"]
grados = list(range(1, 13))
r2_por_grado = []
for g in grados:
    m = make_pipeline(PolynomialFeatures(g), StandardScaler(), LinearRegression())
    r2_por_grado.append(cross_val_score(m, X_mes, y_est, cv=5, scoring="r2").mean())

mes_ord = pd.DataFrame({"mes_llegada": range(1, 13)})
plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.scatter(est["mes_llegada"], est["precio_noche"], s=8, alpha=0.3, color="gray", label="Reservas estándar")
for g, c in [(2, "#e63946"), (8, "#2a6f97")]:
    m = make_pipeline(PolynomialFeatures(g), StandardScaler(), LinearRegression()).fit(X_mes, y_est)
    plt.plot(mes_ord, m.predict(mes_ord), color=c, lw=2, label=f"Grado {g}")
plt.title("Precio de la estándar según el mes")
plt.xlabel("Mes de llegada")
plt.ylabel("Precio por noche (€)")
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(grados, r2_por_grado, marker="o", color="#2a6f97")
plt.title("R² (validación cruzada) según el grado")
plt.xlabel("Grado del polinomio")
plt.ylabel("R²")
plt.tight_layout()
plt.show()

### ✏️ A10 · EJERCICIO 4 de 4 — Polinomio sobre el mes o variable temporada_alta

**Tu turno:** Compara con validación cruzada (5 folds, `scoring="r2"`) sobre las habitaciones estándar dos formas de usar la época del año y guarda las medias en: - `r2_grado2`: polinomio de grado 2 sobre `mes_llegada` (con `make_pipeline(PolynomialFeatures(2), LinearRegression())`), - `r2_alta`: regresión lineal simple sobre la variable `temporada_alta` (una sola columna).

**Debe salir:** `Grado 2 sobre el mes: R² = 0.09 · temporada_alta (1 columna): R² = 0.65` y ✔ Ejercicio 4 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A10).

1. ¿Qué R² salen en cada caso?
2. ¿Por qué una sola columna 0/1 supera a un polinomio de grado 2 sobre el mes?
3. Compara con el polinomio de grado 12 de la gráfica anterior (R² ≈ 0,60): ¿qué concluyes?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A10, **después de intentarlo**.

In [ ]:
# pista: cross_val_score(modelo, X, y, cv=5, scoring="r2").mean() dos veces con distintos X (estas filas son de `est`)
r2_grado2 = ...
r2_alta = ...

assert r2_alta > r2_grado2, "La variable temporada_alta debería rendir más que un polinomio de grado 2 sobre el mes"
assert r2_alta > 0.5, "temporada_alta debería explicar más de la mitad de la variación del precio de la estándar"
print(f"Grado 2 sobre el mes: R² = {r2_grado2:.2f} · temporada_alta (1 columna): R² = {r2_alta:.2f}")
print("✔ Ejercicio 4 correcto")

### ✏️ A11 · MINI-RETO — Función precio_recomendado

**Tu turno:** Con `modelo_lin` (o `con_interacciones`), prepara la función `precio_recomendado(tipo, alta)` que devuelva el precio previsto (€) para un tipo de habitación (`"estandar"`, `"superior"`, `"suite"`, `"familiar"`) y un valor de `alta` (0 o 1). Usa `modelo_lin`.

**Debe salir:** `Estándar en baja: 67 € · Suite en alta: 191 €` y ✔ Mini-reto correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A11).

1. ¿Qué precios salen para una estándar en baja y una suite en alta?
2. ¿Por qué la fila debe tener las mismas columnas y en el mismo orden que `X`?
3. Prueba `precio_recomendado("familiar", 0)`: ¿qué sale?

In [ ]:
# pista: construye una fila con las mismas columnas que X (mismo orden) y usa modelo_lin.predict
def precio_recomendado(tipo, alta):
    ...

p_std_baja = precio_recomendado("estandar", 0)
p_suite_alta = precio_recomendado("suite", 1)

assert 60 < p_std_baja < 80, "Una estándar en temporada baja ronda 70 €"
assert p_suite_alta > 180, "Una suite en temporada alta debería superar los 180 €"
print(f"Estándar en baja: {p_std_baja:.0f} € · Suite en alta: {p_suite_alta:.0f} €")
print("✔ Mini-reto correcto")

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas y los datos

- **Qué hace:**
    - Carga las herramientas: `pandas`, `numpy`, `matplotlib`, `seaborn` y, de `scikit-learn`, la regresión lineal, el modelo *baseline*, `PolynomialFeatures`, el escalador, los *pipelines* y las métricas (MAE, RMSE, R²).
    - `dato(...)` busca un archivo de datos del curso y, si no está, lo descarga del repositorio. `SEMILLA = 42` fija el azar.
    - Lee `reservas_turisdata.csv` (6000 reservas), añade `mes_llegada` y `temporada_alta` (1 en enero, febrero, marzo, julio, agosto y diciembre; 0 en el resto).
- **Qué debe salir:**
    - `(6000, 20)` (6000 reservas, 20 columnas) y `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'pd' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 1. La idea: una recta (o un plano) que se ajusta a los datos

#### A2 · Ejemplo — Preparar X e y con variables dummy

La regresión lineal supone que el precio es una **suma ponderada** de las variables:

`precio ≈ base + (efecto de suite)·es_suite + (efecto de temporada alta)·temporada_alta + …`

Cada **coeficiente** dice *cuántos euros cambia el precio al aumentar esa variable en una unidad, manteniendo las demás iguales*. Por eso es tan apreciada: se puede **explicar**. Las categorías (tipo de habitación) se convierten en columnas 0/1 (*dummies*). `drop_first=True` quita una categoría (`estandar`), que actúa como **referencia**: los coeficientes de las otras se leen como diferencia respecto a la habitación estándar.

- **Qué hace:**
    - `pd.get_dummies(df[["tipo_habitacion"]], drop_first=True)` crea una columna 0/1 por cada tipo de habitación, menos la de referencia; `.astype(int)` las pasa a números enteros.
    - `X["temporada_alta"] = ...` añade la temporada alta como cuarta columna; `y` es el precio por noche, lo que queremos predecir.
    - `X.head()` enseña las 5 primeras filas.
- **Qué debe salir:**
    - Una tabla de 5 filas con 4 columnas: `tipo_habitacion_familiar`, `tipo_habitacion_suite`, `tipo_habitacion_superior` y `temporada_alta`.
- **Si falla:**
    - `NameError: name 'df' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas columnas tiene `X` y cuáles son?* → **4**: `tipo_habitacion_familiar`, `tipo_habitacion_suite`, `tipo_habitacion_superior` y `temporada_alta`.
2. *¿Qué habitación es la «referencia» y cómo se reconoce en la tabla?* → La **estándar**: no tiene columna y se reconoce porque sus tres columnas de tipo valen 0 (por ejemplo, las filas 0, 1, 2 y 4).
3. *¿Qué reserva es la fila 3?* → Una habitación **familiar** (`tipo_habitacion_familiar = 1`) en **temporada alta** (`temporada_alta = 1`).

</details>

#### A3 · Ejercicio 1 — Ajustar la regresión lineal

- **Qué hace:**
    - `train_test_split(X, y, test_size=0.2, random_state=SEMILLA)` reserva el 20 % para prueba (1200 de las 6000 reservas) y fija el reparto con la semilla.
    - `LinearRegression().fit(X_train, y_train)` ajusta la recta (el plano) con train; `modelo_lin.predict(X_test)` predice el precio de las reservas de test (`pred`).
    - `r2_score` y `mean_absolute_error` comparan `pred` con la realidad (`y_test`): `r2_test` y `mae_test`.
- **Qué debe salir:**
    - `R² = 0.915 · MAE = 7.24 € · RMSE = 9.70 €`.
    - `✔ Ejercicio 1 correcto`.
- **Si falla:**
    - `Con 6000 filas y test_size=0.2 el test debe tener 1200 filas` → falta `test_size=0.2` o no has reasignado las cuatro variables.
    - `R² debería superar 0,85` → el modelo no usa las dos variables (tipo de habitación y temporada) o se ha ajustado con otra tabla.
    - `El MAE debería estar entre 5 y 15 euros` → `mae_test` no se ha calculado sobre test.
    - `NameError: name 'pred' is not defined` → falta `pred = modelo_lin.predict(X_test)`: la línea del `print` lo necesita.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué valores salen de R², MAE y RMSE?* → **R² = 0,915**, **MAE = 7,24 €** y **RMSE = 9,70 €**.
2. *¿Qué significa un R² de 0,915?* → Que solo con el tipo de habitación y la temporada, el modelo **explica el 91,5 % de la variación** del precio.
3. *¿Por qué el RMSE (9,70) es mayor que el MAE (7,24)?* → Porque el RMSE eleva los errores al cuadrado y **castiga más los grandes**: hay algunos errores grandes (sobre todo en las suites) que el MAE promedia con los pequeños.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA)
modelo_lin = LinearRegression().fit(X_train, y_train)
pred = modelo_lin.predict(X_test)
r2_test = r2_score(y_test, pred)
mae_test = mean_absolute_error(y_test, pred)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A4 · Ejemplo — El baseline: predecir siempre la media

¿Mejor que «no hacer nada»? El **baseline** de regresión predice siempre la media del precio. Si el modelo no lo supera claramente, no aporta nada.

- **Qué hace:**
    - `DummyRegressor(strategy="mean")` es el modelo tonto: predice siempre la media del precio de train.
    - El `print` escribe su MAE y su R² en test.
- **Qué debe salir:**
    - `Baseline: MAE = 25.56 € · R² = -0.002`.
- **Si falla:**
    - `NameError: name 'X_train' is not defined` → falta el ejercicio anterior (o su solución en la Parte B).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué MAE y qué R² tiene el baseline?* → **MAE = 25,56 €** y **R² = −0,002**.
2. *¿Por qué su R² es ≈ 0?* → Porque predecir siempre la media **no explica nada de la variación**: R² = 0 significa justo «tan bien como la media» (sale algo negativo en test por pura casualidad).
3. *¿Cuánto mejora el modelo lineal frente al baseline?* → El MAE baja de 25,56 € a 7,24 €: **18 € menos de error (un 72 %)**, y el R² pasa de 0 a 0,915.

</details>

## 2. Interpretar los coeficientes

#### A5 · Ejercicio 2 — Leer los coeficientes en euros

- **Qué hace:**
    - `modelo_lin.coef_` es un array con un coeficiente por columna de `X`, en el mismo orden. `pd.Series(..., index=X.columns)` les pone nombre.
    - `coefs["tipo_habitacion_suite"]` y `coefs["temporada_alta"]` sacan los dos que pide el enunciado.
    - `modelo_lin.intercept_` es la **ordenada en el origen**: el precio cuando todas las columnas valen 0, es decir, una estándar en temporada baja.
- **Qué debe salir:**
    - `✔ Ejercicio 2 correcto`. `extra_suite` ≈ 101,2 € y `efecto_temporada_alta` ≈ 22,4 €.
    - La tabla de coeficientes y `Ordenada en el origen (...): 67.3 €`.
- **Si falla:**
    - `Una suite cuesta unos 100 € más que una estándar` → `extra_suite` no es el coeficiente `tipo_habitacion_suite` (¿has cogido otra posición?).
    - `La temporada alta añade unos 20-25 €` → `efecto_temporada_alta` no es el coeficiente de `temporada_alta`.
    - `NameError: name 'modelo_lin' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto valen `extra_suite` y `efecto_temporada_alta`?* → **101,2 €** y **22,4 €**.
2. *¿Qué significa la ordenada en el origen, 67,3?* → El precio previsto de una habitación **estándar en temporada baja**: es la referencia sobre la que se suman los efectos.
3. *Según el modelo, ¿cuánto costaría una familiar en temporada alta?* → 67,3 + 45,0 + 22,4 = **134,7 €** (base + efecto familiar + efecto temporada alta).

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
coefs = pd.Series(modelo_lin.coef_, index=X.columns)
extra_suite = coefs["tipo_habitacion_suite"]
efecto_temporada_alta = coefs["temporada_alta"]
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 3. Residuos: ¿el modelo se equivoca de forma «sana»?

#### A6 · Ejemplo — Residuos del modelo lineal

El **residuo** es `real − predicho`. Si el modelo está bien planteado, los residuos se reparten **sin patrón** alrededor de 0. Si vemos curvas o embudos, falta algo en el modelo.

- **Qué hace:**
    - `residuos = y_test - pred` calcula el error de cada reserva de test, con su signo.
    - A la izquierda, un gráfico de dispersión: precio predicho (horizontal) frente a residuo (vertical), con una línea en 0. A la derecha, un histograma de los residuos.
- **Qué debe salir:**
    - Dos gráficos: a la izquierda una nube en forma de **embudo** (se abre hacia los precios altos); a la derecha una campana centrada cerca de 0.
- **Cómo leerlo:**
    - Se ve un **embudo**: los errores crecen con el precio (la suite falla más, en euros, que la estándar).
    - Tiene sentido: en estos datos las variaciones de precio son **proporcionales** (la temporada alta multiplica ×1,25 y el «ruido» es un % del precio), pero el modelo lineal **suma** efectos fijos en euros.
    - Una parte se puede corregir con **interacciones** (que el efecto de la temporada dependa del tipo de habitación).
- **Si falla:**
    - `NameError: name 'pred' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué forma tiene la nube de la izquierda?* → Un **embudo**: estrecho para los precios bajos y cada vez más abierto hacia los altos.
2. *¿Dónde se equivoca más el modelo, en euros: en las estándar o en las suites?* → **En las suites**: la desviación del residuo es de ≈ 18 € en las suites y de ≈ 7 € en las estándar.
3. *¿Cómo es la distribución de la derecha?* → Una campana centrada **en torno a 0** (media ≈ 0,15 €) y con desviación ≈ 9,7 €: el modelo no se desvía sistemáticamente hacia arriba ni hacia abajo, pero el tamaño del error depende del precio.

</details>

## 4. Términos polinómicos y de interacción

#### A7 · Ejemplo — Interacciones con validación cruzada

**Idea.** `PolynomialFeatures` crea columnas nuevas: potencias (`x²`, `x³`) y productos entre variables (`x1·x2`, las **interacciones**). El modelo sigue siendo lineal *en los coeficientes*, pero puede dibujar curvas. Con `interaction_only=True` solo crea productos.

- **Qué hace:**
    - `PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)` añade a las 4 columnas sus 6 productos de dos en dos (por ejemplo, `suite × temporada_alta`); `make_pipeline(..., LinearRegression())` lo encadena con el modelo.
    - `cross_val_score(..., cv=5, scoring="r2")` calcula el R² en 5 validaciones cruzadas para el modelo lineal y para el de interacciones; `.mean()` promedia.
    - El `print` compara las dos medias.
- **Qué debe salir:**
    - `R² CV lineal: 0.924 · con interacciones: 0.933`.
- **Si falla:**
    - `NameError: name 'X' is not defined` → falta ejecutar el ejemplo que prepara `X` e `y`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué R² medio sale con cada modelo?* → **0,924** el lineal y **0,933** el de interacciones.
2. *¿Cuánto mejora el R² y qué te parece?* → **0,009** (menos de un punto): una mejora **pequeña**, porque el modelo lineal ya explicaba el 92 %.
3. *¿Qué permite una interacción como `suite × temporada_alta`?* → Que el efecto de la temporada alta **dependa del tipo de habitación**: en los datos, la alta añade ≈ 18 € a una estándar y ≈ 38 € a una suite.

</details>

#### A8 · Ejercicio 3 — Error de las interacciones en test

- **Qué hace:**
    - `con_interacciones.fit(X_train, y_train)` ajusta el pipeline (columnas de interacción + regresión lineal) solo con train.
    - `mean_absolute_error(y_test, con_interacciones.predict(X_test))` mide su error en test.
    - `mejora_mae = mae_test - mae_inter_test` es cuántos euros se gana: positivo = mejor.
- **Qué debe salir:**
    - `MAE sin interacciones: 7.24 € · con interacciones: 6.76 € · mejora: 0.48 €`.
    - `✔ Ejercicio 3 correcto`. Es decir, `mae_inter_test` ≈ 6,76 € y `mejora_mae` ≈ 0,48 €.
- **Si falla:**
    - `Las interacciones deberían reducir el error` → has medido el error con otro modelo o has ajustado sin `.fit(X_train, y_train)`.
    - `La mejora debería ser apreciable (más de 0,3 €)` → `mejora_mae` está al revés (restar `mae_inter_test - mae_test` da negativo).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto valen `mae_inter_test` y `mejora_mae`?* → **≈ 6,76 €** y **≈ 0,48 €**.
2. *¿Es mucha mejora? Compárala con el MAE del modelo lineal (7,24 €).* → Es **un 7 %** menos de error: útil pero modesta. Las interacciones corrigen **solo una parte** del embudo.
3. *¿Por qué se ajusta con `X_train` y se mide el error en `X_test`?* → Porque el error de test mide lo que el modelo hará con reservas **que no ha visto**; medirlo con train sería demasiado optimista.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
con_interacciones.fit(X_train, y_train)
mae_inter_test = mean_absolute_error(y_test, con_interacciones.predict(X_test))
mejora_mae = mae_test - mae_inter_test
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 5. Polinomios para captar curvas... y su peligro

#### A9 · Ejemplo — Polinomios de grado 1 a 12

Ahora estudiamos solo las habitaciones **estándar** y cómo cambia el precio con el **mes** (variable numérica de 1 a 12). El precio sube en invierno y verano y baja en primavera y otoño: una relación con curvas. Un polinomio de grado bajo no la capta; uno de grado alto sí, pero se vuelve complejo y **puede sobreajustar**.

- **Qué hace:**
    - `est` se queda con las reservas de habitación estándar (2961); `X_mes` es el mes y `y_est` el precio.
    - El bucle `for g in grados` calcula, para cada grado de 1 a 12, el R² medio en validación cruzada de un polinomio sobre el mes (`PolynomialFeatures` + `StandardScaler` + `LinearRegression`).
    - A la izquierda se dibujan las reservas y las curvas de grado 2 (roja) y grado 8 (azul); a la derecha, el R² según el grado.
- **Qué debe salir:**
    - Dos gráficos. R² por grado: 0,03 · 0,09 · 0,08 · 0,13 · 0,47 · 0,47 · 0,57 · 0,56 · 0,58 · 0,59 · 0,59 · 0,60 (de grado 1 a 12).
- **Si falla:**
    - `NameError: name 'df' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué R² tienen el polinomio de grado 2 y el de grado 8?* → **≈ 0,09** el de grado 2 y **≈ 0,56** el de grado 8.
2. *¿Por qué un polinomio de grado 2 no capta la relación entre el mes y el precio?* → Porque el precio tiene **dos subidas** (invierno y verano) y una parábola solo tiene una curva: no puede subir, bajar, subir y bajar.
3. *¿Qué pasa con el R² al subir el grado de 7 a 12?* → Sigue subiendo, pero **muy poco** (de 0,57 a 0,60): cada grado extra añade complejidad y fragilidad para una ganancia pequeña.

</details>

#### A10 · Ejercicio 4 — Polinomio sobre el mes o variable temporada_alta

- **Qué hace:**
    - `make_pipeline(PolynomialFeatures(2), LinearRegression())` es un polinomio de grado 2; se evalúa sobre `est[["mes_llegada"]]` con validación cruzada de 5 folds y R².
    - `LinearRegression()` sobre `est[["temporada_alta"]]` es una regresión lineal simple con **una sola columna** 0/1.
    - `.mean()` promedia los 5 R² de cada uno.
- **Qué debe salir:**
    - `Grado 2 sobre el mes: R² = 0.09 · temporada_alta (1 columna): R² = 0.65`.
    - `✔ Ejercicio 4 correcto`.
- **Cómo leerlo:**
    - **Lección:** pensar bien las variables (*feature engineering*: aquí «temporada alta» en lugar del mes crudo) casi siempre supera a añadir complejidad al modelo. Un polinomio de grado alto imita la curva, pero es más frágil y difícil de explicar.
- **Si falla:**
    - `La variable temporada_alta debería rendir más que un polinomio de grado 2 sobre el mes` → alguno de los dos R² está mal calculado: ¿usas `est` y no `df`? ¿`temporada_alta` va como una sola columna `[[...]]`?
    - `temporada_alta debería explicar más de la mitad de la variación` → R² ≤ 0,5: el modelo no está bien ajustado (revisa `cv=5` y `scoring="r2"`).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué R² salen en cada caso?* → **0,09** con el polinomio de grado 2 sobre el mes y **0,65** con la columna `temporada_alta`.
2. *¿Por qué una sola columna 0/1 supera a un polinomio de grado 2 sobre el mes?* → Porque `temporada_alta` **ya contiene el patrón real** (enero–marzo, julio–agosto y diciembre son caros): es conocimiento puesto en una columna, no una curva suave.
3. *Compara con el polinomio de grado 12 de la gráfica anterior (R² ≈ 0,60): ¿qué concluyes?* → Que **ni con 12 grados** el polinomio alcanza lo que logra una variable bien pensada (0,65). Es mejor que el modelo sea sencillo y la variable, buena.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
r2_grado2 = cross_val_score(make_pipeline(PolynomialFeatures(2), LinearRegression()),
                            est[["mes_llegada"]], y_est, cv=5, scoring="r2").mean()
r2_alta = cross_val_score(LinearRegression(), est[["temporada_alta"]], y_est, cv=5, scoring="r2").mean()
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 6. Mini-reto: tarifa de referencia explicada al cliente

#### A11 · Mini-reto — Función precio_recomendado

- **Qué hace:**
    - Crea una fila de una tabla con **las mismas columnas que `X`, en el mismo orden**, todas a 0.
    - Pone un 1 en la columna del tipo de habitación pedido (si es la estándar, no hay columna: queda todo a 0) y el valor de `alta` en `temporada_alta`.
    - `modelo_lin.predict(fila)` devuelve el precio previsto; `float(...[0])` lo deja como un número.
- **Qué debe salir:**
    - `Estándar en baja: 67 € · Suite en alta: 191 €`.
    - `✔ Mini-reto correcto`.
- **Si falla:**
    - `Una estándar en temporada baja ronda 70 €` → la fila no tiene las columnas de `X` o están en otro orden.
    - `Una suite en temporada alta debería superar los 180 €` → no has puesto el 1 en `tipo_habitacion_suite` o `alta` no llega a `temporada_alta`.
    - `KeyError` o `ValueError: feature names...` → los nombres de columna no coinciden con los de `X`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué precios salen para una estándar en baja y una suite en alta?* → **67 €** y **191 €** (los de la ordenada en el origen, 67,3, y 67,3 + 101,2 + 22,4).
2. *¿Por qué la fila debe tener las mismas columnas y en el mismo orden que `X`?* → Porque el modelo aprendió un coeficiente **por columna y por posición**: con otro orden, aplicaría el coeficiente de la suite a otra variable. Para la estándar todas valen 0 porque es la referencia.
3. *Prueba `precio_recomendado("familiar", 0)`: ¿qué sale?* → **≈ 112 €** (67,3 de base + 45,0 del efecto familiar).

</details>

## 7. Para reflexionar

1. El coeficiente de `temporada_alta` es ≈ 22 €. ¿Por qué no es el mismo "extra" para una estándar que para una suite?
2. ¿Qué te dice el embudo en los residuos? ¿Cómo lo corregirías (interacciones, transformar el objetivo con logaritmo…)?
3. Un polinomio de grado 12 tiene R² alto en train. ¿Qué mirarías para saber si sobreajusta?
4. ¿Qué información **no** está en el modelo y que un revenue manager usaría para fijar precios (competencia, ocupación, eventos…)?

<details><summary>Ver respuestas (después de pensarlo)</summary>

1. Porque en estos datos la temporada alta **multiplica** el precio (×1,25) en lugar de sumar una cantidad fija: sube más cuanto más caro es el cuarto. En los datos, la alta añade ≈ 18 € a una estándar, ≈ 28 € a una familiar y ≈ 38 € a una suite; el modelo lineal da un único valor medio (22,4 €) a todos.
2. Que el **error crece con el precio** (los residuos miden ≈ 18 € en las suites y ≈ 7 € en las estándar): el modelo no capta que los efectos son proporcionales. Se corrige con interacciones (tipo × temporada), con el **logaritmo del precio** (los efectos pasan a ser porcentajes) o con un modelo por tipo de habitación.
3. Miraría el R² en **validación cruzada o en test**, no en train: en train siempre mejora al subir el grado. Si la puntuación en datos no vistos deja de mejorar o empeora, hay sobreajuste. También desconfiaría de curvas muy onduladas.
4. La **competencia** (precios de hoteles cercanos), la **ocupación** del momento, los **eventos** (fiestas, congresos), la antelación, el día de la semana, las valoraciones o el tiempo previsto.

</details>

## 8. Resumen: qué has aprendido
- Regresión lineal = suma ponderada; los coeficientes se leen en unidades del objetivo.
- MAE (€), RMSE y R² miden el error; el baseline (media) da la referencia.
- Los residuos revelan lo que el modelo no capta; las interacciones y los polinomios lo mejoran.
- Una variable bien pensada vence a más complejidad.